# 强化学习教程：基于 CartPole 的深度 Q 网络（DQN）

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/reinforcelearning/blob/main/rl_tutorial.ipynb)

本教程将带你用 **PyTorch** 从零实现一个 **深度 Q 网络（Deep Q-Network, DQN）**，解决 Gymnasium 中经典的 **CartPole（倒立摆）** 控制问题。

## 你将学到什么

- 强化学习的核心概念：智能体、环境、状态、动作、奖励、策略
- DQN 的基本原理：Q 学习、神经网络近似、贝尔曼方程
- 三大稳定性技巧：**经验回放（Replay Buffer）**、**目标网络（Target Network）**、**软更新（Polyak averaging）**
- ε-greedy 策略下的探索（exploration）与利用（exploitation）平衡
- 完整的 **训练 → 可视化 → 评估** 流程

## 强化学习速览

强化学习是机器学习的一个分支：**智能体（agent）** 在 **环境（environment）** 中执行 **动作（action）**，根据收到的 **奖励（reward）** 信号不断改进自己的 **策略（policy）**，目标是最大化长期累积奖励。

| 概念 | 在 CartPole 中的含义 |
| --- | --- |
| 状态 (state) | 小车位置、小车速度、杆角度、杆角速度（4 维向量） |
| 动作 (action) | 向左推车（0）或向右推车（1） |
| 奖励 (reward) | 杆每多坚持一个时间步 +1 |
| 终止条件 | 杆倒下、小车出界，或达到最大步数 500 |

任务目标：让杆尽量长时间保持直立。在 CartPole-v1 中，每回合最多 500 步，平均奖励 ≥ 475 通常被视为"解决了任务"。

## 1. 环境设置与依赖

我们需要以下几个库：

- `gymnasium`：提供 CartPole 等强化学习环境
- `torch`：搭建并训练 Q 网络
- `numpy` / `matplotlib`：数值计算与可视化

In [ ]:
!pip install -q "gymnasium[classic_control]" torch matplotlib numpy

In [ ]:
import random
from collections import deque, namedtuple
from itertools import count

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# 设备选择：优先 CUDA，其次 Apple MPS，最后 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"使用设备：{device}")

# 设置随机种子以提高可复现性
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. 经验回放缓冲区（Replay Buffer）

DQN 在与环境交互过程中会产生大量 **转移（transition）**：`(state, action, reward, next_state, done)`。如果直接按时间顺序使用这些样本训练，会有两个问题：

1. **样本之间高度相关**：连续帧的状态几乎一样，违反神经网络训练的 i.i.d. 假设。
2. **数据利用率低**：每条经验只用一次就扔掉。

**经验回放缓冲区** 把过往经验存到一个固定容量的队列中，每次训练时从中 **随机采样** 一个 mini-batch，既打破时间相关性，又能多次重用数据，让训练更稳定、更高效。

In [ ]:
Transition = namedtuple("Transition", ("state", "action", "reward", "next_state", "done"))


class ReplayBuffer:
    """固定容量、先进先出的经验回放缓冲区。"""

    def __init__(self, capacity: int):
        self.memory = deque(maxlen=capacity)

    def push(self, *args):
        self.memory.append(Transition(*args))

    def sample(self, batch_size: int):
        return random.sample(self.memory, batch_size)

    def __len__(self) -> int:
        return len(self.memory)

## 3. Q 网络（Q-Network）

Q 网络是一个函数近似器 $Q_\theta(s, a)$，输入 **状态 $s$**，输出 **每个动作的 Q 值**——即在状态 $s$ 下采取动作 $a$ 后能获得的期望累积折扣奖励。

对于 CartPole 这种低维输入，一个简单的多层感知机（MLP）就足够了：

In [ ]:
class DQN(nn.Module):
    """一个 3 层全连接网络，用作 Q(s, ·) 的近似器。"""

    def __init__(self, n_observations: int, n_actions: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_observations, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

## 4. DQN 智能体

`Agent` 类封装了 DQN 算法的核心逻辑：

- **两个网络**：
  - `policy_net`：实时更新，用于选择动作。
  - `target_net`：参数滞后更新，用于计算目标 Q 值，避免"自举"过程中目标剧烈漂移导致训练不稳。
- **ε-greedy 动作选择**：以概率 $\varepsilon$ 随机探索，以概率 $1-\varepsilon$ 贪婪选择当前 Q 值最大的动作。$\varepsilon$ 随训练进度按指数衰减。
- **学习目标（贝尔曼方程）**：

  $$y = r + \gamma \cdot \max_{a'} Q_{\text{target}}(s', a') \cdot (1 - \text{done})$$

  我们最小化 $Q_{\text{policy}}(s, a)$ 与 $y$ 之间的 **Huber Loss**（比 MSE 对离群点更鲁棒）。
- **软更新（Polyak averaging）**：$\theta_{\text{target}} \leftarrow \tau\,\theta_{\text{policy}} + (1-\tau)\,\theta_{\text{target}}$，相比原始 DQN 中"每隔 N 步硬拷贝"的做法更平滑、更稳定。

In [ ]:
class Agent:
    def __init__(self, state_dim: int, action_dim: int):
        # ---- 超参数 ----
        self.state_dim = state_dim
        self.action_dim = action_dim

        self.batch_size = 128        # 每次更新采样的经验数
        self.gamma = 0.99            # 折扣因子
        self.eps_start = 0.9         # ε 起始值
        self.eps_end = 0.05          # ε 最终值
        self.eps_decay = 1000        # ε 衰减步数（越大衰减越慢）
        self.tau = 0.005             # 目标网络软更新系数
        self.lr = 1e-4               # 学习率
        self.grad_clip = 100.0       # 梯度裁剪阈值

        # ---- 网络与优化器 ----
        self.policy_net = DQN(state_dim, action_dim).to(device)
        self.target_net = DQN(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.AdamW(self.policy_net.parameters(), lr=self.lr, amsgrad=True)
        self.memory = ReplayBuffer(capacity=10_000)
        self.steps_done = 0

    # -------------------------------------------------------------
    # 动作选择：ε-greedy
    # -------------------------------------------------------------
    def select_action(self, state: torch.Tensor, greedy: bool = False) -> torch.Tensor:
        eps_threshold = self.eps_end + (self.eps_start - self.eps_end) * \
            np.exp(-1.0 * self.steps_done / self.eps_decay)
        self.steps_done += 1

        if greedy or random.random() > eps_threshold:
            with torch.no_grad():
                return self.policy_net(state).argmax(dim=1, keepdim=True)
        return torch.tensor(
            [[random.randrange(self.action_dim)]], device=device, dtype=torch.long
        )

    # -------------------------------------------------------------
    # 一次 mini-batch 的梯度更新
    # -------------------------------------------------------------
    def optimize_model(self) -> float | None:
        if len(self.memory) < self.batch_size:
            return None

        transitions = self.memory.sample(self.batch_size)
        batch = Transition(*zip(*transitions))

        state_batch = torch.cat(batch.state)
        action_batch = torch.cat(batch.action)
        reward_batch = torch.cat(batch.reward)
        done_batch = torch.cat(batch.done)

        # 终止状态的 next_state 为 None，需要构造掩码
        non_final_mask = ~done_batch
        non_final_next_states = torch.cat(
            [s for s, d in zip(batch.next_state, batch.done) if not d.item()]
        ) if non_final_mask.any() else torch.empty(0, self.state_dim, device=device)

        # 当前 Q 值 Q(s, a)
        state_action_values = self.policy_net(state_batch).gather(1, action_batch)

        # 目标 Q 值 r + γ · max_a' Q_target(s', a')
        next_state_values = torch.zeros(self.batch_size, device=device)
        if non_final_mask.any():
            with torch.no_grad():
                next_state_values[non_final_mask] = self.target_net(non_final_next_states).max(1).values
        expected_state_action_values = reward_batch + self.gamma * next_state_values

        loss = F.smooth_l1_loss(state_action_values, expected_state_action_values.unsqueeze(1))

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(self.policy_net.parameters(), self.grad_clip)
        self.optimizer.step()
        return loss.item()

    # -------------------------------------------------------------
    # 软更新目标网络：θ_target ← τ·θ_policy + (1-τ)·θ_target
    # -------------------------------------------------------------
    def soft_update_target(self) -> None:
        policy_state = self.policy_net.state_dict()
        target_state = self.target_net.state_dict()
        for key in policy_state:
            target_state[key] = self.tau * policy_state[key] + (1.0 - self.tau) * target_state[key]
        self.target_net.load_state_dict(target_state)

## 5. 训练循环

训练逻辑：

1. 重置环境，拿到初始状态。
2. 循环每个时间步：
   - 用 ε-greedy 选一个动作。
   - 在环境中执行该动作，获得 `next_state`、`reward`、`done`。
   - 把这条经验存入回放缓冲区。
   - 调用 `optimize_model` 做一次梯度更新。
   - 对目标网络做一次软更新。
3. 回合结束时记录持续步数（即累积奖励）。

> **训练耗时提示**：在 CPU 上 200 个回合大约需要 1–3 分钟；GPU 会更快。

In [ ]:
def train(num_episodes: int = 400, max_steps: int = 500, log_every: int = 20):
    env = gym.make("CartPole-v1")
    state, _ = env.reset(seed=SEED)
    n_observations = len(state)
    n_actions = env.action_space.n

    agent = Agent(n_observations, n_actions)
    episode_rewards: list[float] = []
    losses: list[float] = []

    for i_episode in range(num_episodes):
        state, _ = env.reset()
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        ep_reward = 0.0

        for t in count():
            action = agent.select_action(state)
            observation, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            ep_reward += float(reward)

            reward_t = torch.tensor([reward], device=device, dtype=torch.float32)
            done_t = torch.tensor([terminated], device=device, dtype=torch.bool)
            next_state = (
                None
                if terminated
                else torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
            )

            # next_state 在终止时用零向量占位，掩码会过滤掉它
            stored_next = next_state if next_state is not None else torch.zeros_like(state)
            agent.memory.push(state, action, reward_t, stored_next, done_t)

            state = stored_next

            loss = agent.optimize_model()
            if loss is not None:
                losses.append(loss)

            # 每一步都做一次软更新（系数 τ 很小，所以变化非常缓慢）
            agent.soft_update_target()

            if done or t + 1 >= max_steps:
                episode_rewards.append(ep_reward)
                break

        if (i_episode + 1) % log_every == 0:
            recent = np.mean(episode_rewards[-log_every:])
            eps_now = agent.eps_end + (agent.eps_start - agent.eps_end) * \
                np.exp(-1.0 * agent.steps_done / agent.eps_decay)
            print(
                f"回合 {i_episode + 1:3d}/{num_episodes} | "
                f"最近 {log_every} 回合平均奖励: {recent:6.2f} | "
                f"ε≈{eps_now:.3f}"
            )

    env.close()
    print("\n训练完成！")
    return agent, episode_rewards, losses


agent, episode_rewards, losses = train(num_episodes=400)

## 6. 训练曲线可视化

我们绘制两条曲线：

- **原始每回合奖励**（浅色）：噪声大，但能反映单回合的波动。
- **滑动平均奖励**（深色，窗口 = 20）：更能反映学习趋势。

如果智能体确实学到了东西，滑动平均应当呈上升趋势，并最终稳定在较高水平。

In [ ]:
def moving_average(x: list[float], window: int = 20) -> np.ndarray:
    if len(x) < window:
        return np.array([])
    return np.convolve(x, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左：每回合奖励 + 滑动平均
ax = axes[0]
ax.plot(episode_rewards, alpha=0.35, label="每回合奖励")
ma = moving_average(episode_rewards, window=20)
if len(ma) > 0:
    ax.plot(range(len(ma)), ma, color="C1", linewidth=2, label="滑动平均（窗口=20）")
ax.axhline(y=475, color="green", linestyle="--", alpha=0.6, label="求解阈值 (475)")
ax.set_xlabel("回合")
ax.set_ylabel("累积奖励 / 持续步数")
ax.set_title("训练进度")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

# 右：损失曲线
ax = axes[1]
ax.plot(losses, alpha=0.4)
ax.set_xlabel("梯度更新次数")
ax.set_ylabel("Huber Loss")
ax.set_title("训练损失")
ax.set_yscale("log")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 评估训练好的智能体

训练完之后，我们关掉探索（ε=0，纯贪婪策略），让智能体跑若干回合，看看真实表现。

In [ ]:
def evaluate(agent: Agent, num_episodes: int = 10, max_steps: int = 500) -> list[float]:
    env = gym.make("CartPole-v1")
    agent.policy_net.eval()
    rewards = []

    for ep in range(num_episodes):
        state, _ = env.reset(seed=SEED + 1000 + ep)
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        ep_reward = 0.0
        for _ in range(max_steps):
            action = agent.select_action(state, greedy=True)
            observation, reward, terminated, truncated, _ = env.step(action.item())
            ep_reward += float(reward)
            if terminated or truncated:
                break
            state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
        rewards.append(ep_reward)
        print(f"评估回合 {ep + 1:2d}: 奖励 = {ep_reward:.0f}")

    env.close()
    agent.policy_net.train()
    print(f"\n平均奖励：{np.mean(rewards):.2f} ± {np.std(rewards):.2f}")
    return rewards


eval_rewards = evaluate(agent, num_episodes=10)

## 8. 进一步优化方向

如果想让效果更好或者把这套代码迁移到更复杂的任务，可以试试：

| 方向 | 做法 | 收益 |
| --- | --- | --- |
| **Double DQN** | 用 `policy_net` 选动作，用 `target_net` 估值 | 缓解 Q 值过估计 |
| **Dueling DQN** | 把网络拆成 状态价值 V(s) + 优势 A(s,a) | 加速收敛 |
| **Prioritized Replay** | 按 TD 误差大小加权采样 | 关注难学样本 |
| **n-step Returns** | 用 n 步回报代替 1 步回报 | 偏差/方差更好的折中 |
| **Rainbow DQN** | 把上面几种组合起来 | DeepMind 当年 SOTA |
| **更大网络 / 更多回合** | 处理 Atari 等高维输入时几乎必需 | — |

> 对于连续动作空间（比如 Pendulum、MuJoCo 机器人控制），DQN 并不适用，应该换成 **DDPG / TD3 / SAC** 等基于策略梯度的算法。

## 参考资料

- Mnih et al., *Human-level control through deep reinforcement learning*, Nature 2015
- Sutton & Barto, *Reinforcement Learning: An Introduction*（强化学习圣经）
- [Gymnasium 官方文档](https://gymnasium.farama.org/)
- [PyTorch DQN 教程](https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html)